In [2]:
import yfinance as yf
import pandas as pd
import io
import os
import time
import requests

In [4]:
df = pd.read_csv('../core/StockTable.csv', sep='\t')
df['Ticker'] = df['Symbol'].str.replace('$', '', regex=False).str.strip()

# 2. Drop AGFS entirely (Taken private in 2023, no public data exists)
df = df[df['Ticker'] != 'AGFS'].copy()

# 3. Define the expanded dictionary of outdated tickers
ticker_updates = {
    # Original list
    'FB': 'META',    # Facebook to Meta
    'PCLN': 'BKNG',  # Priceline to Booking
    'RDS-B': 'SHEL', # Royal Dutch Shell to Shell
    'UTX': 'RTX',    # United Tech to Raytheon
    'CELG': 'BMY',   # Celgene acquired by Bristol-Myers
    
    # The Mergers & Acquisitions
    'BBL': 'BHP',    # BHP Billiton unified structure
    'UN': 'UL',      # Unilever NV merged with PLC
    'HRG': 'SPB',    # HRG Group acquired by Spectrum Brands
    'PICO': 'DHI',   # PICO -> VWTR -> Acquired by D.R. Horton
    
    # The OTC Delistings (New 5-letter tickers)
    'PTR': 'PTRCY',  # PetroChina
    'SNP': 'SNPMF',  # Sinopec
    'ABB': 'ABBNY',  # ABB Ltd.
    'CHL': 'CHLKF',  # China Mobile
}

# 4. Apply the updates to the Symbol column
df['Ticker'] = df['Ticker'].replace(ticker_updates)

# 5. Extract the clean column into a Python list
tickers_list = df['Ticker'].tolist()
tickers_list

['XOM',
 'SHEL',
 'PTRCY',
 'CVX',
 'TOT',
 'BP',
 'BHP',
 'SNPMF',
 'SLB',
 'BHP',
 'AAPL',
 'PG',
 'BUD',
 'KO',
 'PM',
 'TM',
 'PEP',
 'UL',
 'UL',
 'MO',
 'JNJ',
 'PFE',
 'NVS',
 'UNH',
 'MRK',
 'AMGN',
 'MDT',
 'ABBV',
 'SNY',
 'BMY',
 'AMZN',
 'BABA',
 'WMT',
 'CMCSA',
 'HD',
 'DIS',
 'MCD',
 'CHTR',
 'UPS',
 'BKNG',
 'NEE',
 'DUK',
 'D',
 'SO',
 'NGG',
 'AEP',
 'PCG',
 'EXC',
 'SRE',
 'PPL',
 'IEP',
 'SPB',
 'CODI',
 'REX',
 'SPLP',
 'DHI',
 'GMRE',
 'BCH',
 'BSAC',
 'BRK-B',
 'JPM',
 'WFC',
 'BAC',
 'V',
 'C',
 'HSBC',
 'MA',
 'GE',
 'MMM',
 'BA',
 'HON',
 'RTX',
 'LMT',
 'CAT',
 'GD',
 'DHR',
 'ABBNY',
 'GOOG',
 'MSFT',
 'META',
 'T',
 'CHLKF',
 'ORCL',
 'TSM',
 'VZ',
 'INTC',
 'CSCO']

In [48]:
tickers_list = [str(t).replace('$', '').strip() for t in tickers_list]
tickers_list = list(set(tickers_list))
all_price_data = []
failed_tickers = []

print(f"Starting download for {len(tickers_list)} tickers in Stealth Mode...\n")

for i, ticker in enumerate(tickers_list):
    print(f"[{i+1}/{len(tickers_list)}] Downloading {ticker}...")
    
    try:
        # Download 10 years of data using the Ticker object
        stock = yf.Ticker(ticker)
        df_hist = stock.history(period='10y')
        
        # Check if Yahoo actually returned data (handles dead tickers)
        if not df_hist.empty:
            df_hist['Symbol'] = ticker
            
            # The 'Date' is currently an index. Let's make it a normal column.
            df_hist = df_hist.reset_index()
            
            # THE BULLETPROOF DATE FIX: Strip timezone AND normalize to midnight
            df_hist['Date'] = pd.to_datetime(df_hist['Date']).dt.tz_localize(None).dt.normalize()
            
            all_price_data.append(df_hist)
        else:
            print(f"  -> Warning: No data found for {ticker}")
            failed_tickers.append(ticker)
            
    except Exception as e:
        print(f"  -> Error downloading {ticker}: {e}")
        failed_tickers.append(ticker)
        
    # Pause for 1 full second to respect rate limits and avoid the shadow ban!
    time.sleep(1)

print("\n--- Download Loop Finished ---")

# Combine all individual dataframes into one massive master dataframe
master_price_df = pd.concat(all_price_data, ignore_index=True)

# Save using pyarrow
save_path = '/Users/lawrence/Library/CloudStorage/GoogleDrive-lawrencemrmay@gmail.com/My Drive/_Uni/Stock_preds_LLM/data/stocknet_10y_prices.parquet'
master_price_df.to_parquet(save_path, engine='pyarrow')

print(f"Success! Saved {len(master_price_df)} total rows of historical data.")
if failed_tickers:
    print(f"Failed to download: {failed_tickers}")

# Let's look at those beautiful, clean dates!
display(master_price_df.head())

Starting download for 85 tickers in Stealth Mode...

[1/85] Downloading MCD...
[2/85] Downloading AEP...
[3/85] Downloading TSM...
[4/85] Downloading CSCO...
[5/85] Downloading PTRCY...


$PTRCY: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for PTRCY
[6/85] Downloading BP...
[7/85] Downloading META...
[8/85] Downloading DIS...
[9/85] Downloading TOT...
[10/85] Downloading D...
[11/85] Downloading WFC...
[12/85] Downloading CVX...
[13/85] Downloading TM...
[14/85] Downloading AAPL...
[15/85] Downloading INTC...
[16/85] Downloading UNH...
[17/85] Downloading GD...
[18/85] Downloading CMCSA...
[19/85] Downloading PEP...
[20/85] Downloading REX...
[21/85] Downloading BKNG...
[22/85] Downloading ABBNY...
[23/85] Downloading SHEL...
[24/85] Downloading BA...
[25/85] Downloading HON...
[26/85] Downloading SPB...
[27/85] Downloading MA...
[28/85] Downloading SO...
[29/85] Downloading BCH...
[30/85] Downloading JPM...
[31/85] Downloading ABBV...
[32/85] Downloading PFE...
[33/85] Downloading IEP...
[34/85] Downloading AMZN...
[35/85] Downloading C...
[36/85] Downloading EXC...
[37/85] Downloading MO...
[38/85] Downloading ORCL...
[39/85] Downloading NVS...
[40/85] Downloading BSAC...
[41/85] Downloading

$CHLKF: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for CHLKF
[49/85] Downloading HD...
[50/85] Downloading SLB...
[51/85] Downloading SNPMF...
[52/85] Downloading DHI...
[53/85] Downloading DHR...
[54/85] Downloading VZ...
[55/85] Downloading KO...
[56/85] Downloading UL...
[57/85] Downloading AMGN...
[58/85] Downloading MSFT...
[59/85] Downloading GE...
[60/85] Downloading SRE...
[61/85] Downloading BMY...
[62/85] Downloading PG...
[63/85] Downloading BABA...
[64/85] Downloading V...
[65/85] Downloading BUD...
[66/85] Downloading CODI...
[67/85] Downloading MDT...
[68/85] Downloading NEE...
[69/85] Downloading DUK...
[70/85] Downloading SPLP...


$SPLP: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for SPLP
[71/85] Downloading UPS...
[72/85] Downloading WMT...
[73/85] Downloading BAC...
[74/85] Downloading PCG...
[75/85] Downloading MRK...
[76/85] Downloading BHP...
[77/85] Downloading JNJ...
[78/85] Downloading T...
[79/85] Downloading PM...
[80/85] Downloading SNY...
[81/85] Downloading BRK-A...
[82/85] Downloading MMM...
[83/85] Downloading CAT...
[84/85] Downloading LMT...
[85/85] Downloading RTX...

--- Download Loop Finished ---



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.

In [1]:
master_price_df = pd.read_csv('/Users/lawrence/Library/CloudStorage/GoogleDrive-lawrencemrmay@gmail.com/My Drive/_Uni/Stock_preds_LLM/data/stocknet_10y_prices.csv'

SyntaxError: incomplete input (1723464178.py, line 1)

In [36]:
# # Create an empty list to hold each ticker's dataframe
# all_price_data = []
# failed_tickers = []

# print(f"Starting download for {len(tickers_list)} tickers...\n")

# for i, ticker in enumerate(tickers_list):
#     print(f"[{i+1}/{len(tickers_list)}] Downloading {ticker}...")
    
#     try:
#         # Download 10 years of data
#         stock = yf.Ticker(ticker)
#         df_hist = stock.history(period='10y')
        
#         # Check if Yahoo actually returned data (handles dead tickers)
#         if not df_hist.empty:
#             # CRITICAL: Add the ticker symbol to the dataframe!
#             df_hist['Symbol'] = ticker
            
#             # The 'Date' is currently an index. Let's make it a normal column.
#             df_hist = df_hist.reset_index()
            
#             # Ensure the Date column is timezone-naive to avoid merging issues later
#             df_hist['Date'] = pd.to_datetime(df_hist['Date']).dt.tz_localize(None)
            
#             all_price_data.append(df_hist)
#         else:
#             print(f"  -> Warning: No data found for {ticker}")
#             failed_tickers.append(ticker)
            
#     except Exception as e:
#         print(f"  -> Error downloading {ticker}: {e}")
#         failed_tickers.append(ticker)
        
#     # Pause for half a second to respect rate limits
#     time.sleep(0.5)

# print("\n--- Download Loop Finished ---")

# # Combine all individual dataframes into one massive master dataframe
# master_price_df = pd.concat(all_price_data, ignore_index=True)

# master_price_df.to_parquet('/Users/lawrence/Library/CloudStorage/GoogleDrive-lawrencemrmay@gmail.com/My Drive/_Uni/Stock_preds_LLM/data/stocknet_10y_prices.parquet')

# print(f"Success! Saved {len(master_price_df)} total rows of historical data.")
# if failed_tickers:
#     print(f"Failed to download: {failed_tickers}")

Starting download for 87 tickers...

[1/87] Downloading XOM...
[2/87] Downloading SHEL...
[3/87] Downloading PTRCY...


$PTRCY: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for PTRCY
[4/87] Downloading CVX...
[5/87] Downloading TOT...
[6/87] Downloading BP...
[7/87] Downloading BHP...
[8/87] Downloading SNPMF...
[9/87] Downloading SLB...
[10/87] Downloading BHP...
[11/87] Downloading AAPL...
[12/87] Downloading PG...
[13/87] Downloading BUD...
[14/87] Downloading KO...
[15/87] Downloading PM...
[16/87] Downloading TM...
[17/87] Downloading PEP...
[18/87] Downloading UL...
[19/87] Downloading UL...
[20/87] Downloading MO...
[21/87] Downloading JNJ...
[22/87] Downloading PFE...
[23/87] Downloading NVS...
[24/87] Downloading UNH...
[25/87] Downloading MRK...
[26/87] Downloading AMGN...
[27/87] Downloading MDT...
[28/87] Downloading ABBV...
[29/87] Downloading SNY...
[30/87] Downloading BMY...
[31/87] Downloading AMZN...
[32/87] Downloading BABA...
[33/87] Downloading WMT...
[34/87] Downloading CMCSA...
[35/87] Downloading HD...
[36/87] Downloading DIS...
[37/87] Downloading MCD...
[38/87] Downloading CHTR...
[39/87] Downloading UP

$SPLP: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for SPLP
[56/87] Downloading DHI...
[57/87] Downloading GMRE...
[58/87] Downloading BCH...
[59/87] Downloading BSAC...
[60/87] Downloading BRK-A...
[61/87] Downloading JPM...
[62/87] Downloading WFC...
[63/87] Downloading BAC...
[64/87] Downloading V...
[65/87] Downloading C...
[66/87] Downloading HSBC...
[67/87] Downloading MA...
[68/87] Downloading GE...
[69/87] Downloading MMM...
[70/87] Downloading BA...
[71/87] Downloading HON...
[72/87] Downloading RTX...
[73/87] Downloading LMT...
[74/87] Downloading CAT...
[75/87] Downloading GD...
[76/87] Downloading DHR...
[77/87] Downloading ABBNY...
[78/87] Downloading GOOG...
[79/87] Downloading MSFT...
[80/87] Downloading META...
[81/87] Downloading T...
[82/87] Downloading CHLKF...


$CHLKF: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


  -> Warning: No data found for CHLKF
[83/87] Downloading ORCL...
[84/87] Downloading TSM...
[85/87] Downloading VZ...
[86/87] Downloading INTC...
[87/87] Downloading CSCO...

--- Download Loop Finished ---



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Success! Saved 208740 total rows of historical data.
Failed to download: ['PTRCY', 'SPLP', 'CHLKF']
